<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260326_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lexical Scope, Scope Chain, 클로저를 활용한 정보 은닉(Encapsulation)

## 1. 배경

자바스크립트 엔진이 코드를 실행할 때 가장 먼저 해결해야 하는 과제는 변수의 유효 범위와 수명을 관리하는 것입니다. 초기 자바스크립트는 전역 변수의 남용으로 인해 데이터 오염이 빈번했으나, 엔진의 발전과 함께 실행 컨텍스트(Execution Context) 내에서 식별자를 찾는 메커니즘이 정교해졌습니다.

- 식별자 결정의 효율성: 함수가 어디서 호출되었는가가 아니라 어디서 정의되었는가에 따라 상위 스코프를 결정하는 규칙이 정립되었습니다.
- 데이터 보안: 자바스크립트는 타 언어와 달리 별도의 private 접근 제한자가 없었기에, 함수의 스코프 특성을 이용해 외부로부터 데이터를 보호하는 기법이 필수적으로 요구되었습니다.
- 효율적인 메모리 관리: 함수 실행이 끝난 후에도 필요한 데이터를 유지해야 하는 현대 웹 애플리케이션의 요구사항을 충족하기 위해 클로저 메커니즘이 핵심 역할을 하게 되었습니다.

## 2. 설명

### 2.1. Lexical Scope

렉시컬 스코프는 함수를 어디서 호출했는지가 아니라, 함수를 어디서 정의했는지에 따라 상위 스코프가 결정되는 정적 스코프 규칙을 의미합니다.

- 결정 시점: 코드가 작성(컴파일)되는 시점에 스코프가 확정됩니다.
- 특징: 자바스크립트의 모든 함수는 자신의 정의된 환경(상위 스코프)을 기억합니다.

### 2.2. Scope Chain

스코프 체인은 식별자를 찾기 위해 현재 스코프에서 시작하여 상위 스코프 방향으로 이동하며 검색하는 물리적인 연결 리스트를 의미합니다.

- 식별자 검색: 현재 스코프에 변수가 없으면 외부 렉시컬 환경(Outer Lexical Environment)으로 거슬러 올라가며 검색합니다.
- 종착지: 최상위 스택인 전역 스코프에서도 변수를 찾지 못하면 ReferenceError를 발생시킵니다.

### 2.3. 클로저를 활용한 정보 은닉 (Encapsulation)

클로저는 외부 함수보다 중첩 함수가 더 오래 유지되는 경우, 이미 실행이 종료된 외부 함수의 변수를 참조할 수 있는 중첩 함수를 의미합니다.

- 정보 은닉: 외부에서 직접 접근할 수 없는 변수를 함수 내부에 숨기고, 오직 클로저 함수를 통해서만 해당 변수를 조작하게 만듭니다.
- 캡슐화: 데이터와 데이터를 조작하는 메서드를 하나로 묶어 결합도를 높이고 의도치 않은 변경을 방지합니다.

## 3. 예제

### 3.1. Lexical Scope 확인

In [17]:
%%writefile ex1.js

const globalName = "김철수"; // 전역 스코프 — 어디서든 접근 가능

function outerFn() {
    const outerName = "이영희"; // outerFn 스코프 — outerFn 안에서만 접근 가능

    function innerFn() {
        // innerFn은 자신의 스코프에 globalName, outerName이 없으면
        // 바깥쪽 스코프로 올라가서 찾음 (스코프 체인)
        // 이때 "어디서 호출됐냐"가 아니라 "어디서 정의됐냐" 기준으로 찾음
        // 이게 바로 Lexical Scope (정적 스코프)
        console.log(globalName); // 전역에서 찾음 -> "김철수"
        console.log(outerName);  // outerFn 스코프에서 찾음 -> "이영희"
    }

    innerFn();
}

outerFn();

Overwriting ex1.js


In [2]:
!node ex1.js

김철수
이영희


3.2. 클로저를 통한 기본 카운터

In [3]:
%%writefile ex2.js

function createCounter() {
    // count는 외부에서 직접 접근 불가 — 클로저로 보호
    // createCounter가 끝나도 반환된 함수가 count를 기억하고 있음
    let count = 0;

    // 이 함수가 count를 배낭에 챙겨서 밖으로 나감
    return function() {
        count++;      // 호출할 때마다 count를 1씩 증가
        return count; // 증가된 값 반환
    };
}

// parkCounter는 count를 기억하는 함수
const parkCounter = createCounter();

console.log(parkCounter()); // 1
console.log(parkCounter()); // 2

// 새로 만들면 count가 0부터 다시 시작 — 완전히 독립적
const leeCounter = createCounter();
console.log(leeCounter());  // 1 — parkCounter와 별개
console.log(parkCounter()); // 3 — parkCounter는 계속 이어서

Writing ex2.js


In [4]:
!node ex2.js

1
2


3.3. 정보 은닉과 객체 반환 (Encapsulation)

In [5]:
%%writefile ex3.js

function userManager(initialName) {
    // 외부에서 직접 접근 불가 — 클로저로 보호되는 private 변수
    let userName = initialName;

    // 외부에 공개할 메서드만 객체로 반환
    return {
        // userName을 읽기만 하는 메서드 (getter)
        getName: function() {
            return userName;
        },
        // userName을 변경하는 메서드 (setter)
        // 이 메서드를 통해서만 값 변경 가능
        setName: function(newName) {
            userName = newName;
        }
    };
}

const choiManager = userManager("최현우");

console.log(choiManager.getName()); // "최현우" — getter로 접근
choiManager.setName("최현우_최종"); // setter로만 변경 가능
console.log(choiManager.getName()); // "최현우_최종"
console.log(choiManager.userName);  // undefined — 직접 접근 불가

Writing ex3.js


In [6]:
!node ex3.js

최현우
최현우_최종
undefined




---
## 4. 실습 및 모범 답안

### 실습 1: 스코프 체인 탐색 이해

문제: 아래 코드에서 `console.log(value)`의 출력 결과와 그 이유를 스코프 체인 관점에서 서술하세요.

In [7]:
%%writefile ex4.js

const value = "전역";       // 전역 스코프 — 가장 바깥

function firstFn() {
    const value = "첫번째"; // firstFn 스코프 — 전역 value를 가림(shadowing)

    function secondFn() {
        // 1. 먼저 자기 자신(secondFn) 스코프에서 value 찾기 -> 없음
        // 2. 바로 위 firstFn 스코프에서 value 찾기 -> "첫번째" 발견!
        // 3. 전역까지 올라갈 필요 없이 여기서 탐색 종료
        // 이게 스코프 체인 — 안쪽에서 바깥쪽으로 올라가며 순서대로 찾음
        console.log(value); // "첫번째"
    }

    secondFn();
}

firstFn();

Writing ex4.js


In [18]:
!node ex4.js

첫번째


상세 설명: secondFn 내부에는 value가 정의되어 있지 않습니다. 따라서 스코프 체인을 통해 바로 위 상위 스코프인 firstFn의 렉시컬 환경을 검색합니다. firstFn에 value가 존재하므로 검색을 멈추고 해당 값을 출력합니다. 전역 스코프까지 올라가지 않는 이유는 하위에서 상위로의 검색 원칙 때문입니다.



---

### 실습 2: 클로저를 활용한 데이터 독립성 유지

문제: `createStudent` 함수를 만들어 학생의 점수를 관리하세요. 점수는 외부에서 직접 수정할 수 없어야 하며, `addScore`와 `getScore` 메서드를 통해서만 조작해야 합니다.

In [21]:
%%writefile ex5.js

function createStudent(name) {
    // score는 외부에서 직접 접근 불가 — 클로저로 보호
    // name도 마찬가지로 외부에서 직접 수정 불가
    let score = 0;

    return {
        // 점수를 추가하는 메서드 — score에 접근하는 유일한 창구
        addScore(point) {
            score += point;
        },
        // 현재 점수를 확인하는 메서드 — score를 읽기만 함
        getScore() {
            return `${name}의 점수: ${score}점`;
        }
    };
}

const limStudent = createStudent("임윤아");

limStudent.addScore(10);
console.log(limStudent.getScore()); // "임윤아의 점수: 10점"

limStudent.addScore(20);
console.log(limStudent.getScore()); // "임윤아의 점수: 30점"

// 외부에서 직접 접근 시도
console.log(limStudent.score); // undefined — 직접 접근 불가

Overwriting ex5.js


In [22]:
!node ex5.js

임윤아의 점수: 10점
임윤아의 점수: 30점
undefined


상세 설명: score 변수는 함수의 실행이 끝나도 힙 메모리에 유지됩니다. 반환된 객체의 메서드들이 클로저로서 이 환경을 기억하기 때문에, 임윤아 학생만의 독립적인 점수 관리가 가능해집니다.



---

### 실습 3: 렉시컬 스코프의 정적 특징 확인

문제: 다음 코드를 실행했을 때 출력되는 결과가 "강하늘"인지 "이진우"인지 예측하고 그 이유를 서술하세요.

In [11]:
%%writefile ex6.js

const myName = "강하늘"; // 전역 스코프

function printName() {
    // printName은 전역에서 정의됐음
    // 호출된 위치(wrapper 안)가 아니라 정의된 위치 기준으로 스코프 결정
    // -> 전역의 myName = "강하늘" 을 참조
    console.log(myName);
}

function wrapper() {
    const myName = "이진우"; // wrapper 스코프 — printName과 무관
    printName(); // printName을 여기서 호출해도 스코프는 바뀌지 않음
}

wrapper();

Writing ex6.js


In [12]:
!node ex6.js

강하늘


상세 설명: 자바스크립트는 렉시컬 스코프를 따릅니다. printName 함수는 wrapper 안에서 호출되었지만, 정의된 위치는 전역입니다. 따라서 printName의 상위 스코프는 호출 시점의 환경이 아닌 정의 시점의 환경인 전역 스택이 됩니다. 전역에 선언된 myName인 "강하늘"이 출력되는 것이 올바른 동작입니다.



---
### 실습 4: 클로저를 이용한 캡슐화와 유효성 검사

문제: 은행 계좌를 관리하는 `bankAccount` 함수를 작성하세요. `deposit` 메서드는 양수일 때만 입금이 가능해야 하며, 잔액(balance)은 외부에서 직접 수정할 수 없어야 합니다.


In [26]:
%%writefile ex7.js

function bankAccount(initialBalance) {
    // balance는 외부에서 직접 접근 불가 — 클로저로 보호
    // initialBalance로 초기값 설정 후 내부에서만 관리
    let balance = initialBalance;

    return {
        // 입금 메서드 — 유효성 검사 후 balance 증가
        deposit(amount) {
            // 0이하 금액은 입금 거부
            if (amount <= 0) {
                console.log("입금액은 양수여야 합니다.");
                return;
            }
            // 유효한 금액이면 balance에 더함
            balance += amount;
            console.log(`${amount}원 입금 완료. 현재 잔액: ${balance}원`);
        },
        // 잔액 확인 메서드 — balance를 읽기만 함
        getBalance() {
            return `현재 잔액: ${balance}원`;
        }
    };
}

const myAccount = bankAccount(1000);

myAccount.deposit(500);              // "500원 입금 완료. 현재 잔액: 1500원"
console.log(myAccount.getBalance()); // "현재 잔액: 1500원"

// 유효성 검사 확인
myAccount.deposit(-100);             // "입금액은 양수여야 합니다."

// 외부에서 직접 접근 시도
console.log(myAccount.balance);      // undefined — 직접 접근 불가

Overwriting ex7.js


In [27]:
!node ex7.js

500원 입금 완료. 현재 잔액: 1500원
현재 잔액: 1500원
입금액은 양수여야 합니다.
undefined


상세 설명: balance라는 핵심 데이터를 은닉하고 deposit이라는 인터페이스를 통해서만 접근하게 함으로써, 잘못된 데이터(음수 입금 등)가 들어오는 것을 방지하는 객체지향적 캡슐화를 클로저로 구현한 예시입니다.



---

### 실습 5: 클로저와 반복문 (고전적 문제 해결)

문제: 아래 코드는 0, 1, 2가 아닌 3, 3, 3을 출력합니다. 클로저나 블록 스코프의 특성을 활용하여 0, 1, 2가 순차적으로 출력되도록 수정하세요.

In [30]:
%%writefile ex8.js

function timerTest() {
    // var 대신 let 사용 — 반복마다 독립적인 i가 새로 생성됨
    // var였다면 루프가 끝난 후 i = 3이 된 상태에서 콜백 3개가 전부 실행됨
    for (let i = 0; i < 3; i++) {
        // 이 setTimeout은 100ms 후에 실행되지만
        // let 덕분에 각 반복의 i값(0, 1, 2)을 독립적으로 기억함
        setTimeout(function() {
            console.log(i); // 0, 1, 2 순서대로 출력
        }, 100);
    }
}

timerTest(); // 0, 1, 2

Overwriting ex8.js


In [31]:
!node ex8.js

0
1
2


- 상세 설명: `var`는 함수 레벨 스코프를 가지므로 전역에 가깝게 동작하여 모든 타이머가 종료 시점의 `i`인 3을 바라봅니다. `let`을 사용하면 반복마다 새로운 렉시컬 환경이 생성되고, `setTimeout`의 콜백 함수가 해당 시점의 `i`를 기억하는 클로저가 되어 의도한 대로 동작합니다.



---



# Prototype 상속 구조, __proto__와 prototype 속성의 차이 분석

## 1. 배경

자바스크립트는 클래스 기반 언어(Java, C++)와 달리 **프로토타입 기반 객체 지향 언어**입니다. 1995년 설계 당시 메모리 자원을 효율적으로 사용하기 위해, 동일한 메서드를 모든 객체가 복사해서 갖는 대신 하나의 '원형(Prototype)'을 공유하는 방식을 채택했습니다.

- **메모리 최적화:** 수만 개의 객체를 생성해도 공통 메서드는 단 하나만 존재하므로 메모리 낭비를 방지합니다.
- **동적 확장:** 이미 생성된 객체라도 원형에 메서드를 추가하면 즉시 모든 하위 객체에서 사용할 수 있는 유연함을 제공합니다.
- **상속의 구현:** 별도의 클래스 키워드 없이도 객체 간의 연결을 통해 기능을 물려받는 독특한 상속 체계를 구축했습니다.

## 2. 설명

### 2.1. Prototype 상속 구조

자바스크립트의 모든 객체는 자신의 부모 역할을 하는 객체와 연결되어 있습니다. 이를 프로토타입이라고 하며, 이 연결 고리가 끝까지 이어져 상속을 구현하는 것을 **프로토타입 체인**(Prototype Chain)이라고 합니다.

- **검색 메커니즘:** 객체의 속성에 접근할 때 해당 객체에 속성이 없으면 부모 프로토타입으로 올라가며 검색합니다.
- **종착지:** 최상위 부모는 `Object.prototype`이며, 여기서도 찾지 못하면 `undefined`를 반환합니다.

### 2.2. __proto__와 prototype 속성의 차이 분석

이 두 속성은 이름이 비슷하여 가장 혼란을 주는 부분이지만, 주체와 용도가 명확히 다릅니다.

- **prototype 속성:**
    - **주체:** 오직 **함수 객체**(Constructor)만 가집니다.
    - **용도:** 해당 함수를 통해 생성될 **자식 객체들에게 물려줄 원형**을 가리킵니다.
- **__proto__ 속성 (접근자 프로퍼티):**
    - **주체:** **모든 객체**가 가집니다.
    - **용도:** 현재 객체를 있게 한 **실제 부모 객체**를 가리키는 링크입니다. (상속의 결과물)

> **핵심 요약:** `생성자함수.prototype`은 부모가 "내 자식들은 이걸 써라" 하고 내놓은 설계도이고, `자식객체.__proto__`는 자식이 "내 부모는 이 설계도다"라고 가리키는 손가락입니다.
>

## 3. 예제

### 3.1. 생성자 함수를 통한 프로토타입 공유

In [54]:
%%writefile /content/eex1.js

function Person(name) {
    this.name = name;
    // name은 인스턴스마다 따로 저장 — kim은 "김철수", lee는 "이영희"
}

// prototype에 메서드 추가 — 모든 인스턴스가 이 메서드 하나를 공유
// 인스턴스마다 함수를 새로 만들지 않아서 메모리 절약
Person.prototype.sayHello = function() {
    // this = 호출한 인스턴스 (kim이 부르면 this.name = "김철수")
    return `안녕하세요, 저는 ${this.name}입니다.`;
};

const kim = new Person("김철수");
const lee = new Person("이영희");

console.log(kim.sayHello()); // "안녕하세요, 저는 김철수입니다."
console.log(lee.sayHello()); // "안녕하세요, 저는 이영희입니다."

// kim과 lee는 sayHello를 각자 갖고 있는 게 아니라
// 같은 prototype의 함수를 함께 바라보고 있음
console.log(kim.sayHello === lee.sayHello); // true — 같은 메모리 주소

Overwriting /content/eex1.js


In [55]:
!node eex1.js

안녕하세요, 저는 김철수입니다.
안녕하세요, 저는 이영희입니다.
true


3.2. __proto__와 prototype의 실체 확인

In [57]:
%%writefile /content/eex2.js

function Student(name) {
    this.name = name;
}

const park = new Student("박지민");

// park.__proto__ === Student.prototype
// new Student()로 만든 순간 park의 __proto__가 Student.prototype에 자동 연결됨
console.log(park.__proto__ === Student.prototype); // true

// Student 함수 자체도 객체 — Function으로 만들어진 인스턴스
// 그래서 Student.__proto__는 Function.prototype을 가리킴
console.log(Student.__proto__ === Function.prototype); // true

Overwriting /content/eex2.js


In [58]:
!node eex2.js

true
true


3.3. 프로토타입 체인을 이용한 상속

In [60]:
%%writefile /content/eex3.js

const animal = {
    eats: true,
    walk() {
        return "동물이 걷습니다.";
    }
};

const rabbit = {
    jumps: true,
    __proto__: animal // rabbit의 부모를 animal로 직접 지정
};

// rabbit 자신에게 eats가 없음 → __proto__(animal)에서 찾음
console.log(rabbit.eats);   // true

// rabbit 자신에게 walk가 없음 → __proto__(animal)에서 찾음
console.log(rabbit.walk()); // "동물이 걷습니다."

// rabbit 자신의 속성
console.log(rabbit.jumps);  // true — rabbit에 직접 있음

Overwriting /content/eex3.js


In [61]:
!node eex3.js

true
동물이 걷습니다.
true




---

## 4. 실습 및 모범 답안

### 실습 1: 프로토타입 메서드 확장

**문제:** 모든 배열(Array) 객체가 사용할 수 있는 `sum` 메서드를 `Array.prototype`에 추가하세요. 이 메서드는 배열 내 모든 숫자의 합을 반환해야 합니다.

In [40]:
%%writefile eex4.js

// Array.prototype에 sum 메서드 추가
// 이렇게 하면 모든 배열에서 .sum() 사용 가능
Array.prototype.sum = function() {
    // this = sum을 호출한 배열 (여기선 scores)
    let total = 0;
    for (let i = 0; i < this.length; i++) {
        total += this[i]; // 배열 요소를 하나씩 더함
    }
    return total;
};

const scores = [10, 20, 30];
console.log(scores.sum()); // 60

// 다른 배열에서도 사용 가능 — 프로토타입에 추가했으니까
const prices = [100, 200, 300];
console.log(prices.sum()); // 600

Overwriting eex4.js


In [41]:
!node eex4.js

60
600


상세 설명: Array라는 빌트인 생성자 함수의 prototype에 메서드를 추가하면, 이후 생성되는 모든 배열 객체는 __proto__를 통해 이 메서드를 참조할 수 있게 됩니다. 이를 통해 최현우 개발자는 모든 배열에서 공통 로직을 편리하게 호출할 수 있습니다.



---

### 실습 2: __proto__를 통한 객체 간 연결

**문제:** `parentUser` 객체를 만들고, 이를 상속받는 `childUser` 객체를 생성하세요. `childUser`에서 `parentUser`의 속성에 접근하는 과정을 스택/힙 메모리 관점에서 서술하세요.

In [42]:
%%writefile eex5.js

const parentUser = {
    nation: "Korea",
    greet() { return "반갑습니다"; }
};

// Object.create — parentUser를 프로토타입으로 하는 새 객체 생성
const childUser = Object.create(parentUser);

// childUser 자신의 속성 추가
childUser.name = "김철수";
childUser.age  = 25;

// childUser 자신의 속성 접근
console.log(childUser.name);    // "김철수" — childUser 자신에게 있음

// parentUser에서 상속받은 속성 접근
console.log(childUser.nation);  // "Korea" — childUser에 없으니 __proto__ 타고 올라가서 찾음
console.log(childUser.greet()); // "반갑습니다" — 동일하게 프로토타입 체인으로 찾음

// 프로토타입 연결 확인
console.log(childUser.__proto__ === parentUser); // true

Writing eex5.js


In [43]:
!node eex5.js

김철수
Korea
반갑습니다
true


상세 설명: childUser 객체 자체에는 nation 속성이 없습니다. 자바스크립트 엔진은 childUser의 스택 레퍼런스를 따라 힙 메모리로 이동한 뒤 속성을 찾고, 없으면 __proto__ 링크를 따라 부모인 parentUser의 힙 영역을 탐색하여 "Korea"를 찾아냅니다.



---

### 실습 3: 생성자 함수와 인스턴스의 관계 증명

**문제:** 강하늘 학생 객체를 생성자 함수로 만들고, `Object.getPrototypeOf()` 메서드를 사용하여 해당 객체의 부모가 누구인지 출력하세요.

In [46]:
%%writefile eex6.js

function Developer(name) {
    this.name = name;
}

const kang = new Developer("강하늘");

// kang의 프로토타입(부모) 확인
console.log(Object.getPrototypeOf(kang));          // Developer.prototype 객체 출력
console.log(Object.getPrototypeOf(kang) === Developer.prototype); // true — 부모가 Developer.prototype임을 증명

// 추가 확인
console.log(kang instanceof Developer);             // true — kang은 Developer로 만들어진 인스턴스
console.log(kang.name);                             // "강하늘"

Overwriting eex6.js


In [48]:
!node eex6.js

{}
true
true
강하늘


상세 설명: __proto__는 브라우저 하위 호환성을 위한 비표준 접근자였으므로(현재는 표준화되었으나 권장되지 않음), 실제 현업에서는 Object.getPrototypeOf()를 사용하여 정다은 개발자처럼 안전하게 객체의 원형을 추적합니다.



---

### 실습 4: 프로토타입 체인의 종착지 확인

**문제:** 아래 코드에서 최종적으로 `null`이 출력될 때까지 `__proto__`를 추적하는 코드를 작성하고, 왜 마지막에 `null`이 나오는지 서술하세요.

In [49]:
%%writefile eex7.js

const myObj = {};

// 1단계 — myObj의 프로토타입 (Object.prototype)
console.log(Object.getPrototypeOf(myObj));
// { constructor: f, hasOwnProperty: f, ... } — Object.prototype

// 2단계 — Object.prototype의 프로토타입
console.log(Object.getPrototypeOf(Object.getPrototypeOf(myObj)));
// null — 프로토타입 체인의 종착지

// while로 한 번에 끝까지 추적
let current = myObj;
let step = 0;

while (current !== null) {
    console.log(`${step}단계:`, current);
    current = Object.getPrototypeOf(current); // 한 단계 위로
    step++;
}
console.log(`${step}단계: null — 체인 종료`);

Writing eex7.js


In [50]:
!node eex7.js

[Object: null prototype] {}
null
0단계: {}
1단계: [Object: null prototype] {}
2단계: null — 체인 종료


상세 설명: 모든 객체의 뿌리는 Object.prototype입니다. 자바스크립트 설계상 프로토타입 체인의 최상단인 Object.prototype의 부모는 존재하지 않으므로 null로 설정되어 검색의 끝을 알립니다.



---
### 실습 5: 프로토타입 덮어쓰기 주의점

**문제:** 아래 코드에서 `lee` 객체가 `sayHi` 메서드를 호출하지 못하는 이유를 프로토타입 객체 치환 관점에서 설명하세요.


In [53]:
%%writefile eex8.js

function Member(name) {
    this.name = name;
}

// 1. lee 생성 — 이 시점에 lee.__proto__가 "기존 Member.prototype"을 가리킴
const lee = new Member("이진우");

// 2. Member.prototype을 완전히 새 객체로 교체
//    lee는 여전히 "기존 Member.prototype"을 바라보고 있음
//    lee와 새 Member.prototype은 이제 남남
Member.prototype = {
    sayHi() { return "하이"; }
};

// 3. lee는 새 prototype을 모름 — sayHi 없음
try {
    console.log(lee.sayHi());
} catch (e) {
    console.error("에러 발생:", e.message); // "lee.sayHi is not a function"
}

// 4. lee의 __proto__는 기존 객체, Member.prototype은 새 객체 — 서로 다름
console.log("같은가?:", Object.getPrototypeOf(lee) === Member.prototype); // false

Overwriting eex8.js


In [52]:
!node eex8.js

에러 발생: lee.sayHi is not a function
lee의 프로토타입과 Member.prototype이 같은가?: false


상세 설명: lee가 생성되는 시점에 lee.__proto__는 기존의 Member.prototype 객체를 가리킵니다. 이후 Member.prototype을 sayHi가 있는 새로운 객체로 통째로 교체하면, 기존에 생성된 lee는 교체 이전의 프로토타입을 계속 바라보고 있으므로 새로운 객체에 추가된 sayHi를 찾을 수 없습니다. 결과적으로 Object.getPrototypeOf(lee) === Member.prototype이 false가 되며, lee.sayHi is not a function 에러가 발생합니다. 인스턴스 생성 후 프로토타입을 교체할 때는 이 점에 주의가 필요합니다.